# Rotated digit clustering (Assignment 2, Q7)

Selected cells from the original submission. Outputs are cleared; see README for the recorded results and comparison caveat. Run from this folder with the original input arrays present.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler


In [ ]:
def add_username_to_plot(ax, username: str):
        """Adds the username to a plot axis. Useful to avoid repeating code."""
        ax.text(0.98, 0.98, username, ha='right', va='top', transform=ax.transAxes, fontsize=10, color='gray', alpha=0.7)

In [ ]:
import hashlib
# Seed for random generators, as required
# My email ID is aks.kanodia@students.iiit.ac.in

username = "aks.kanodia"
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)

## PCA and K-means implementations

In [ ]:
class KMeansClustering:
    """
    A custom implementation of the K-Means clustering algorithm.

    This class replicates core functionalities like fitting the model to data, predicting cluster assignments, and calculating the cost (WCSS).
    """
    def __init__(self, k=3, max_iters=100, seed=42):
        """
        Initializes the K-Means model.

        Args:
            k (int): The number of clusters to form.
            max_iters (int): The maximum number of iterations for the algorithm.
            seed (int): Seed for the random number generator for reproducibility.
        """
        self.k = k
        self.max_iters = max_iters
        self.seed = seed
        self.centroids = None
        self.clusters = None
        self.cost = None
        self.rng = np.random.default_rng(seed=self.seed)

    def init_centroids(self, X):
        """
        Initializes centroids using Kmeans++ algorithm.

        Args:
            X (np.ndarray): The input dataset of shape (n_samples, n_features).
        """
        n_samples, n_features = X.shape

        centroids = np.zeros((self.k, n_features))
        centroids[0] = X[self.rng.choice(n_samples)]
        for i in range(1, self.k):
            distances_sq = np.min(((X - centroids[:i, np.newaxis])**2).sum(axis=2), axis=0)
            probabilities = distances_sq / np.sum(distances_sq)
            next_centroid_idx = self.rng.choice(n_samples, p=probabilities)
            centroids[i] = X[next_centroid_idx]
        self.centroids = centroids

    def assign_clusters(self, X):
        """
        Assigns each data point to the closest centroid.

        Args:
            X (np.ndarray): The input dataset.

        Returns:
            np.ndarray: An array of cluster indices for each data point.
        """
        distances_sq = ((X - self.centroids[:, np.newaxis])**2).sum(axis=2)
        return np.argmin(distances_sq, axis=0) # can take min of squared distances also...!

    def update_centroids(self, X, cluster_indices):
        """
        Updates centroids by calculating the mean of all data points in a cluster.

        Args:
            X (np.ndarray): The input dataset.
            cluster_indices (np.ndarray): The current cluster assignments for each data point.

        Returns:
            np.ndarray: The new, updated centroids.
        """
        new_centroids = self.centroids.copy()
        for i in range(self.k):
            cluster_points = X[cluster_indices == i]
            if len(cluster_points) > 0:
                new_centroids[i] = np.mean(cluster_points, axis=0)
            # If cluster is empty, doesn't change centroid (let it be)
        return new_centroids

    def fit(self, X):
        """
        Trains the K-Means model by finding the optimal centroids for K clusters.
        The method uses the Expectation-Maximization approach of assigning clusters and then updating centroids iteratively.

        Args:
            X (np.ndarray): The training dataset of shape (n_samples, n_features).
        """
        self.init_centroids(X) # Initialize centroids

        for _ in range(self.max_iters):
            # Expectation step
            self.clusters = self.assign_clusters(X)

            # Maximization step
            new_centroids = self.update_centroids(X, self.clusters)

            # Check convergence (here, if centroids stop moving)
            if np.allclose(self.centroids, new_centroids):
                break

            self.centroids = new_centroids

        # final cost
        self.cost = self.getCost(X)

    def predict(self, X):
        """
        Assigns a cluster number to each data point based on the learned centroids.

        Args:
            X (np.ndarray): The dataset to predict clusters for.

        Returns:
            np.ndarray: The array of predicted cluster indices.
        """
        if self.centroids is None:
            raise RuntimeError("The model has not been fitted yet. Call fit() first.")
        return self.assign_clusters(X) # Can simply perform assignment based on existing centroids

    def getCost(self, X):
        """
        Returns the Within-Cluster Sum of Squares (WCSS), which is the total squared distance between each point and its assigned centroid.

        Args:
            X (np.ndarray): The dataset.

        Returns:
            float: The calculated WCSS value.
        """
        cost = 0
        if self.centroids is None or self.clusters is None:
             raise RuntimeError("The model has not been fitted yet. Call fit() first.")

        for i in range(self.k):
            points_in_cluster = X[self.clusters == i]
            if len(points_in_cluster) > 0:
                cost += np.sum((points_in_cluster - self.centroids[i])**2)
        return cost

In [ ]:
class PCA:
    """
    A class to perform Principal Component Analysis (PCA) from scratch.
    """
    def __init__(self, n_components):
        """
        Initializes the PCA model.

        Parameters:
        n_components (int): The number of principal components to keep.
        """
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ratio = None

    def fit(self, X):
        """
        Fits the PCA model to the data X.

        This method computes the principal components of the dataset.

        Parameters:
        X (np.ndarray): The training data of shape (n_samples, n_features).
        """
        # Center the data (subtract mean)
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean

        # covariance matrix
        cov_matrix = np.cov(X_centered, rowvar=False)

        # Calculate eigenvalues and eigenvectors
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # sort eigenvectors (descending eigenvals)
        # transpose eigenvectors to rows for simplicity
        eigenvectors = eigenvectors.T
        idxs = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]

        # Store the top n_components as principal components
        self.components = eigenvectors[0:self.n_components]

        # Store explained variance for analysis
        total_variance = np.sum(eigenvalues)
        self.explained_variance_ratio = eigenvalues / total_variance # Not just for the components picked, this is for total all components!

    def transform(self, X):
        """
        Transforms the data using the fitted principal components.

        Parameters:
        X (np.ndarray): The data to transform of shape (n_samples, n_features).

        Returns:
        np.ndarray: The transformed data of shape (n_samples, n_components).
        """
        if self.components is None:
            raise RuntimeError("PCA model has not been fitted yet. Call fit() first.")

        # Center the data
        X_centered = X - self.mean

        # Project data onto principal components
        # need to transpose cause I stored as rows not columns
        return np.dot(X_centered, self.components.T)

    def inverse_transform(self, X_transformed):
        """
        Transforms data back to its original space.

        Parameters:
        X_transformed (np.ndarray): The transformed data of shape (n_samples, n_components).

        Returns:
        np.ndarray: The reconstructed data in the original feature space.
        """
        if self.components is None:
            raise RuntimeError("PCA model has not been fitted yet. Call fit() first.")

        # Reconstruct from principal components and add the mean back
        reconstructed_centered = np.dot(X_transformed, self.components)
        return reconstructed_centered + self.mean

    def checkPCA(self, X_test):
        """
        Checks if the class reduces dimensions appropriately.

        Parameters:
        X_test (np.ndarray): A sample dataset to test the transformation.

        Returns:
        bool: True if the transformed data has the correct dimensions, False otherwise.
        """
        self.fit(X_test)
        X_transformed = self.transform(X_test)

        expected_shape = (X_test.shape[0], self.n_components)

        if X_transformed.shape == expected_shape:
            return True
        else:
            # print(f"checkPCA Failed: Transformed shape is {X_transformed.shape}, expected {expected_shape}.")
            return False

## Load and inspect the provided images

In [ ]:
# Load the dataset from the .npy file
X_modified = np.load("X_modified.npy")
print(f"Dataset loaded successfully. Shape: {X_modified.shape}")
y_true = np.load("y_true.npy")

# Select a random image to display (no seed, this is to understand the transformations in general...)
sample_indices = np.random.choice(X_modified.shape[0], size=5, replace=False)
fig, axes = plt.subplots(1, len(sample_indices), figsize=(3 * len(sample_indices), 3))
fig.suptitle("Modified Dataset Given")

for ax, sample_index in zip(axes, sample_indices):
    sample_image = X_modified[sample_index]
    sample_label = y_true[sample_index]

    ax.imshow(sample_image, cmap='gray')
    ax.set_title(f'Label: {sample_label}')
    ax.axis('off')

    # Add username to each subplot
    add_username_to_plot(ax, username)

plt.tight_layout()
plt.show()

## Orientation heuristics

In [ ]:
def apply_rotation(data, border_width=5):
    """
    Individually detects and corrects the orientation of each image by finding the border with the most "ink" (pixel intensity).

    - If the top border is heaviest, the digit is upside down (rotate 180 deg).
    - If the right border is heaviest, the digit is rotated 90 deg CCW (rotate 90 deg CW).
    - If the left border is heaviest, the digit is rotated 90 deg CW (rotate 90 deg CCW).
    - If the bottom border is heaviest, the digit is already upright (no rotation).

    Parameters:
    data (np.ndarray): The dataset of images (n_samples, 28, 28).
    border_width (int): The number of rows/columns to consider as the border.

    Returns:
    np.ndarray: A new dataset with orientation-corrected images.
    """
    corrected_data = []
    for img in data:
        # Calculate the sum of pixel intensities for each border
        border_sums = {}

        border_sums['top'] = np.sum(img[:border_width, :])
        border_sums['bottom'] = np.sum(img[-border_width:, :])
        border_sums['left'] = np.sum(img[:, :border_width])
        border_sums['right'] = np.sum(img[:, -border_width:])

        heaviest_border = max(border_sums, key=border_sums.get)

        # Apply the corresponding rotation
        if heaviest_border == 'top':
            corrected_img = np.rot90(img, k=2)
        elif heaviest_border == 'right':
            corrected_img = np.rot90(img, k=-1)
        elif heaviest_border == 'left':
            corrected_img = np.rot90(img, k=1)
        else:
            corrected_img = img

        corrected_data.append(corrected_img)

    return np.array(corrected_data)

In [ ]:
X_corrected = apply_rotation(X_modified)

# Select a random image to display (no seed, this is to understand the transformations in general...)
# sample_indices = np.random.choice(X_corrected.shape[0], size=5, replace=False)
# Use same samples that were visualized initially
fig, axes = plt.subplots(1, len(sample_indices), figsize=(3 * len(sample_indices), 3))
fig.suptitle("Dataset after correction")

for ax, sample_index in zip(axes, sample_indices):
    sample_image = X_corrected[sample_index]
    sample_label = y_true[sample_index]

    ax.imshow(sample_image, cmap='gray')
    ax.set_title(f'Label: {sample_label}')
    ax.axis('off')

    # Add username to each subplot
    add_username_to_plot(ax, username)

plt.tight_layout()
plt.show()

In [ ]:
def apply_rotation_v2(data, border_width=5):
    """
    Individually detects and corrects the orientation of each image by finding the border with the most "ink" (pixel intensity).

    - If the top border is heaviest, the digit is upside down (rotate 180 deg).
    - If the right border is heaviest, the digit is rotated 90 deg CCW (rotate 90 deg CW).
    - If the left border is heaviest, the digit is rotated 90 deg CW (rotate 90 deg CCW).
    - Ignore bottom border (assume no image is already upright)

    Parameters:
    data (np.ndarray): The dataset of images (n_samples, 28, 28).
    border_width (int): The number of rows/columns to consider as the border.

    Returns:
    np.ndarray: A new dataset with orientation-corrected images.
    """
    corrected_data = []
    for img in data:
        # Calculate the sum of pixel intensities for each border
        border_sums = {}

        border_sums['top'] = np.sum(img[:border_width, :])
        border_sums['left'] = np.sum(img[:, :border_width])
        border_sums['right'] = np.sum(img[:, -border_width:])

        heaviest_border = max(border_sums, key=border_sums.get)

        # Apply the corresponding rotation
        if heaviest_border == 'top':
            corrected_img = np.rot90(img, k=2)
        elif heaviest_border == 'right':
            corrected_img = np.rot90(img, k=-1)
        elif heaviest_border == 'left':
            corrected_img = np.rot90(img, k=1)

        corrected_data.append(corrected_img)

    return np.array(corrected_data)

In [ ]:
X_corrected_v2 = apply_rotation_v2(X_modified)

# Select a random image to display (no seed, this is to understand the transformations in general...)
# sample_indices = np.random.choice(X_corrected_v2.shape[0], size=5, replace=False)
# Using same samples that were initially visualized
fig, axes = plt.subplots(1, len(sample_indices), figsize=(3 * len(sample_indices), 3))
fig.suptitle("Dataset after correction (v2)")

for ax, sample_index in zip(axes, sample_indices):
    sample_image = X_corrected_v2[sample_index]
    sample_label = y_true[sample_index]

    ax.imshow(sample_image, cmap='gray')
    ax.set_title(f'Label: {sample_label}')
    ax.axis('off')

    # Add username to each subplot
    add_username_to_plot(ax, username)

plt.tight_layout()
plt.show()

In [ ]:
# Pick fresh samples for this section
sample_indices = np.random.choice(X_modified.shape[0], size=5, replace=False)

datasets = [
    ("Modified Dataset Given", X_modified),
    ("Dataset after correction", X_corrected),
    ("Dataset after correction (v2)", X_corrected_v2),
]

# Create subplots: 3 rows (datasets) × N columns (images)
fig, axes = plt.subplots(len(datasets), len(sample_indices), figsize=(3 * len(sample_indices), 9))
fig.suptitle(f"Comparison of Transformations | Username : {username}", fontsize=16)

for row, (title, X_data) in enumerate(datasets):
    for col, sample_index in enumerate(sample_indices):
        ax = axes[row, col] if len(datasets) > 1 else axes[col]  # handles 1D vs 2D case
        sample_image = X_data[sample_index]
        sample_label = y_true[sample_index]

        ax.imshow(sample_image, cmap='gray')
        if row == 0:
            ax.set_title(f'Label: {sample_label}')
        ax.axis('off')

    # Row titles on the left side
    fig.text(0.02, (len(datasets)-row-0.75) / len(datasets), title, va='center', ha='left', fontsize=12)

plt.tight_layout(rect=[0.18, 0, 1, 0.95])  # leave space for suptitle
plt.show()

## Original pipeline comparison

The corrected pipelines also use scaling and PCA, unlike the raw baseline. The ARI values below cannot attribute any difference to rotation alone.

In [ ]:
# Instantiate and fit the KMeans models
model_original = KMeansClustering(k=10, seed=seed)
model_corrected = KMeansClustering(k=10, seed=seed)
model_corrected_v2 = KMeansClustering(k=10, seed=seed)

# Flatten the image data before fitting the KMeans models
X_modified_flat = X_modified.reshape(X_modified.shape[0], -1)
X_corrected_flat = X_corrected.reshape(X_corrected.shape[0], -1)
X_corrected_v2_flat = X_corrected_v2.reshape(X_corrected_v2.shape[0], -1)

# Only for corrected datasets
scaler = StandardScaler()
X_corrected_flat = scaler.fit_transform(X_corrected_flat)
X_corrected_v2_flat = scaler.fit_transform(X_corrected_v2_flat)

# Apply PCA as well (only to corrected datasets)
pca = PCA(n_components=150)
pca.fit(X_corrected_flat)
X_corrected_flat = pca.transform(X_corrected_flat)
print("Explained variance of 150 components = ", pca.explained_variance_ratio[:150].sum())
pca.fit(X_corrected_v2_flat)
X_corrected_v2_flat = pca.transform(X_corrected_v2_flat)
print("Explained variance of 150 components = ", pca.explained_variance_ratio[:150].sum())

model_original.fit(X_modified_flat)
y_pred_original_kmeans = model_original.predict(X_modified_flat)

model_corrected.fit(X_corrected_flat)
y_pred_corrected_kmeans = model_corrected.predict(X_corrected_flat)

model_corrected_v2.fit(X_corrected_v2_flat)
y_pred_corrected_v2_kmeans = model_corrected_v2.predict(X_corrected_v2_flat)

# Calculate WCSS (cost) for each model
wcss_original = model_original.getCost(X_modified_flat)
wcss_corrected = model_corrected.getCost(X_corrected_flat)
wcss_corrected_v2 = model_corrected_v2.getCost(X_corrected_v2_flat)

print(f"Original Dataset WCSS: {wcss_original:.4f}")
print(f"Dataset after correction WCSS: {wcss_corrected:.4f}")
print(f"Dataset after correction (v2) WCSS: {wcss_corrected_v2:.4f}")

# Calculate Adjusted Rand Score for clustering performance
ars_original_kmeans = adjusted_rand_score(y_true, y_pred_original_kmeans)
ars_corrected_kmeans = adjusted_rand_score(y_true, y_pred_corrected_kmeans)
ars_corrected_v2_kmeans = adjusted_rand_score(y_true, y_pred_corrected_v2_kmeans)

print(f"\nOriginal Dataset Adjusted Rand Score (KMeans): {ars_original_kmeans:.4f}")
print(f"Dataset after correction Adjusted Rand Score (KMeans): {ars_corrected_kmeans:.4f}")
print(f"Dataset after correction (v2) Adjusted Rand Score (KMeans): {ars_corrected_v2_kmeans:.4f}")